# 02 — Lightweight Customer Segmentation

SQL-based RFM scoring + K-means clustering. Used to check whether forecast drivers
(temperature cold snaps, event weeks) affect customer segments differently.
This is a supporting module — forecasting is the centrepiece.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from src.utils.db_connect import get_engine

engine = get_engine()
print('Connected')

## 1. RFM Metrics from SQL

In [ ]:
rfm_sql = """
WITH rfm_base AS (
    SELECT
        t.customer_id,
        c.customer_type,
        c.nationality,
        c.age_band,
        CAST(
            julianday((SELECT MAX(date) FROM daily_sales)) -
            julianday(MAX(t.date))
        AS INTEGER) AS recency_days,
        COUNT(t.transaction_id) AS frequency,
        ROUND(SUM(t.total_amount), 2) AS monetary
    FROM transactions t
    JOIN customers c ON t.customer_id = c.customer_id
    GROUP BY t.customer_id, c.customer_type, c.nationality, c.age_band
)
SELECT * FROM rfm_base
"""
rfm = pd.read_sql(rfm_sql, engine)
print(f'RFM records: {len(rfm):,}')
rfm.describe()

## 2. K-Means Clustering on R/F/M

In [ ]:
features = rfm[['recency_days', 'frequency', 'monetary']].copy()
scaler = StandardScaler()
X = scaler.fit_transform(features)

# Elbow + silhouette to pick k
inertias, silhouettes = [], []
k_range = range(2, 8)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig = make_subplots(rows=1, cols=2, subplot_titles=['Elbow', 'Silhouette Score'])
fig.add_trace(go.Scatter(x=list(k_range), y=inertias, mode='lines+markers', name='Inertia'), row=1, col=1)
fig.add_trace(go.Scatter(x=list(k_range), y=silhouettes, mode='lines+markers', name='Silhouette'), row=1, col=2)
fig.update_layout(title='K-Means Selection')
fig.show()

In [ ]:
K = 4  # Adjust based on elbow/silhouette above
km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
rfm['cluster'] = km_final.fit_predict(X)

cluster_summary = rfm.groupby('cluster')[['recency_days', 'frequency', 'monetary']].mean().round(1)
print(cluster_summary)

fig = px.scatter_3d(rfm.sample(5000, random_state=42),
                     x='recency_days', y='frequency', z='monetary',
                     color='cluster', opacity=0.5,
                     title='RFM Clusters (3D)',
                     labels={'recency_days': 'Recency (days)',
                              'frequency': 'Frequency', 'monetary': 'Monetary (AUD)'})
fig.show()

## 3. Do Cold Snaps Affect Segments Differently?

In [ ]:
cold_snap_sql = """
SELECT
    CASE WHEN ds.temperature_index < 15 THEN 'Cold snap' ELSE 'Normal' END AS temp_condition,
    t.customer_type,
    COUNT(t.transaction_id) AS num_transactions,
    ROUND(AVG(t.total_amount), 2) AS avg_basket,
    ROUND(AVG(t.num_items), 2) AS avg_items
FROM transactions t
JOIN daily_sales ds ON t.date = ds.date
GROUP BY temp_condition, t.customer_type
"""
cold_snap = pd.read_sql(cold_snap_sql, engine)

fig = px.bar(cold_snap, x='customer_type', y='avg_basket',
             color='temp_condition', barmode='group',
             title='Avg Basket Size: Cold Snap vs Normal Days by Customer Segment',
             labels={'avg_basket': 'Avg Basket (AUD)', 'customer_type': 'Customer Type'})
fig.update_yaxes(tickformat='$,.0f')
fig.show()

## 4. Do Event Weeks Affect Segments Differently?

In [ ]:
event_seg_sql = """
SELECT
    ds.event_flag,
    t.customer_type,
    COUNT(t.transaction_id) AS num_transactions,
    ROUND(AVG(t.total_amount), 2) AS avg_basket,
    ROUND(SUM(t.total_amount), 0) AS total_spend
FROM transactions t
JOIN daily_sales ds ON t.date = ds.date
GROUP BY ds.event_flag, t.customer_type
"""
event_seg = pd.read_sql(event_seg_sql, engine)

fig = px.bar(event_seg, x='event_flag', y='avg_basket',
             color='customer_type', barmode='group',
             title='Avg Basket by Event Type × Customer Segment',
             labels={'avg_basket': 'Avg Basket (AUD)', 'event_flag': 'Event'})
fig.update_yaxes(tickformat='$,.0f')
fig.show()

## 5. Cluster × Customer Type Cross-Tab

In [ ]:
crosstab = pd.crosstab(rfm['cluster'], rfm['customer_type'], normalize='index') * 100
print('Cluster composition by customer type (%):')
print(crosstab.round(1))

fig = px.bar(crosstab.reset_index().melt(id_vars='cluster'),
             x='cluster', y='value', color='customer_type', barmode='stack',
             title='Cluster Composition by Customer Type',
             labels={'value': 'Share (%)', 'cluster': 'Cluster'})
fig.update_yaxes(ticksuffix='%')
fig.show()